# Chapter 6 — PageIndex SDK 실전

> Cloud SaaS · Expert Routing · Chat API · Self-hosted
> v2.0 / 2026 · NOWAVE


## 튜토리얼 구성

| § | 튜토리얼 | 데이터셋 | 학습 포인트 |
|---|---|---|---|
| 6-1 | SDK 1줄 트리 생성 | sample_10k.pdf | submit → poll → get_tree |
| 6-2 | LLM Tree Search 분해 | sample_10k.pdf | 트리 압축 + node_list 추론 |
| 6-3 | **Expert-Guided RAG** ★ | sample_10k.pdf + 한국어 PDF | 도메인 규칙 주입 효과 |
| 6-4 | Chat API + Multi-turn | sample_10k.pdf | OpenAI 키 없이 챗봇 |
| 6-5 | Managed vs Self-hosted 비교 | 두 PDF 모두 | 트리 일관성·지연·비용 |

## API 키 처리 — Cell Skip 패턴

`PAGEINDEX_API_KEY` 환경변수가 없으면 해당 cell은 자동 skip된다. OpenAI 부분만으로도 §6-2의 LLM Tree Search 학습 가능.

## 0. 환경 준비

```bash
pip install -q pageindex openai python-dotenv reportlab pymupdf
```

비용은 약 $1.5~$5 (PageIndex API $0.5~$2 + OpenAI $1~$3).

### 0.1 환경 변수 + 클라이언트 초기화

PAGEINDEX_API_KEY가 없으면 Cloud 관련 셀을 자동 skip한다.

In [2]:
# ─────────────────────────────────────────────────────
# 환경 변수 + 키 가용성 체크
# ─────────────────────────────────────────────────────
import os, json, time
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

# 환경 변수 체크 — Cloud 셀의 skip 조건
PAGEINDEX_API_KEY = os.environ.get("PAGEINDEX_API_KEY")
OPENAI_API_KEY    = os.environ.get("OPENAI_API_KEY")

HAVE_PAGEINDEX = PAGEINDEX_API_KEY is not None and PAGEINDEX_API_KEY != ""
HAVE_OPENAI    = OPENAI_API_KEY is not None

print("환경 변수 상태:")
print(f"  OPENAI_API_KEY:    {'✓' if HAVE_OPENAI else '✗ (필수)'}")
print(f"  PAGEINDEX_API_KEY: {'✓' if HAVE_PAGEINDEX else '✗ (Cloud cell skip)'}")
print()
print("PageIndex 키 발급: https://dash.pageindex.ai/api-keys")

WORK = Path("./work"); WORK.mkdir(exist_ok=True)
RES  = Path("./work/results"); RES.mkdir(parents=True, exist_ok=True)

환경 변수 상태:
  OPENAI_API_KEY:    ✓
  PAGEINDEX_API_KEY: ✓

PageIndex 키 발급: https://dash.pageindex.ai/api-keys


In [3]:
# ─────────────────────────────────────────────────────
# OpenAI + PageIndex 클라이언트 초기화 (조건부)
# ─────────────────────────────────────────────────────
from openai import OpenAI

openai_client = OpenAI() if HAVE_OPENAI else None

pi_client = None
if HAVE_PAGEINDEX:
    try:
        from pageindex import PageIndexClient
        pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)
        print("✓ PageIndex client 준비")
    except ImportError:
        print("pageindex 미설치 — `pip install pageindex` 후 재실행")
        HAVE_PAGEINDEX = False
else:
    print("AGEINDEX_API_KEY 미설정 — Cloud 셀은 모두 skip")

if HAVE_OPENAI:
    print("✓ OpenAI client 준비")

✓ PageIndex client 준비
✓ OpenAI client 준비


---
## 1. 데이터셋 준비 — 2개 PDF

1. **sample_10k.pdf** — Chapter 1에서 만든 영문 SEC 10-K 합성 문서 (재사용)
2. **kepco_ko_report.pdf** — 본 챕터에서 자체 생성하는 한국어 공공문서 (KEPCO 형태)

두 번째 PDF는 책 일관성을 위해 ReportLab + NanumGothic으로 자체 생성한다. Chapter 3 §3-3의 패턴과 동일하다.

In [4]:
# ─────────────────────────────────────────────────────
# 1번 PDF — Chapter 1 산출물 재사용
# ─────────────────────────────────────────────────────
PDF_10K  = WORK / "sample_10k.pdf"
assert PDF_10K.exists(), "Chapter 1 노트북을 먼저 실행하여 sample_10k.pdf를 만드시오"
print(f"✓ {PDF_10K.name}: {PDF_10K.stat().st_size:,} bytes")

✓ sample_10k.pdf: 2,064 bytes


In [5]:
# ─────────────────────────────────────────────────────
# 2번 PDF — 한국어 공공문서 자체 생성 (KEPCO 형태)
# ─────────────────────────────────────────────────────
from reportlab.pdfgen import canvas
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

PDF_KO = WORK / "kepco_ko_report.pdf"

# 한글 폰트 등록 시도 — 시스템 폰트 자동 탐색
korean_font_path = None
for cand in ["/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
             "/Library/Fonts/AppleSDGothicNeo.ttc",
             "C:/Windows/Fonts/malgun.ttf",
             os.path.expanduser("~/.fonts/NanumGothic.ttf")]:
    if Path(cand).exists():
        korean_font_path = cand
        break

if korean_font_path:
    try:
        pdfmetrics.registerFont(TTFont("KOR", korean_font_path))
        FNT = "KOR"
        print(f"한글 폰트 등록: {korean_font_path}")
    except Exception:
        FNT = "Helvetica"
        print("한글 폰트 등록 실패 → Helvetica (한글 깨질 수 있음)")
else:
    FNT = "Helvetica"
    print("한글 폰트 미발견 → Helvetica (한글 깨질 수 있음)")

# KEPCO 형태 한국어 공공문서 섹션 정의
sections = [
    ("한국전력공사 2025년 사업보고서",       20, True),
    ("",                                       11, False),
    ("제 1 장 회사 개요",                     18, True),
    ("1.1 회사 연혁",                         14, True),
    ("당사는 1898년 설립된 한국 최대 전력 공기업이다.", 11, False),
    ("1.2 사업 영역",                         14, True),
    ("핵심 사업은 전력 생산·송전·배전이다.",  11, False),
    ("1.3 조직 구조",                         14, True),
    ("본사 5개 본부와 12개 지역사업본부.",     11, False),
    ("",                                       11, False),
    ("제 2 장 재무 정보",                     18, True),
    ("2.1 매출 현황",                         14, True),
    ("2025년 매출은 76조원으로 전년 대비 8.3% 증가하였다.", 11, False),
    ("2.2 영업 이익",                         14, True),
    ("영업이익은 4.2조원으로 흑자 전환하였다.", 11, False),
    ("2.3 부채 현황",                         14, True),
    ("총 부채는 168조원이며 부채비율 230% 수준이다.", 11, False),
    ("",                                       11, False),
    ("제 3 장 리스크 요인",                   18, True),
    ("3.1 환율 리스크",                        14, True),
    ("원/달러 환율 변동이 연료 수입 비용에 직접 영향.", 11, False),
    ("3.2 연료비 리스크",                      14, True),
    ("국제 유가·LNG 가격 변동에 노출되어 있다.", 11, False),
    ("3.3 규제 리스크",                        14, True),
    ("탄소 배출 규제 강화로 추가 비용 발생 가능.", 11, False),
    ("",                                       11, False),
    ("제 4 장 ESG·지속가능경영",              18, True),
    ("4.1 탄소중립 로드맵",                    14, True),
    ("2050년 탄소중립 달성을 위한 단계적 전환.", 11, False),
    ("4.2 신재생에너지 투자",                  14, True),
    ("향후 5년간 신재생 발전에 30조원 투자 예정.", 11, False),
]

# PDF 작성
c = canvas.Canvas(str(PDF_KO))
y = 760
for text, size, bold in sections:
    if not text:
        y -= 8; continue
    c.setFont(FNT, size)
    c.drawString(60, y, text)
    y -= 28
    if y < 80:
        c.showPage(); y = 760
c.save()
print(f"\n✓ {PDF_KO.name}: {PDF_KO.stat().st_size:,} bytes")

한글 폰트 미발견 → Helvetica (한글 깨질 수 있음)

✓ kepco_ko_report.pdf: 2,507 bytes


---
## §6-1 SDK 1줄 트리 생성

`PageIndexClient`로 PDF 업로드 → polling → 트리 페치의 3단계를 실행한다. 환경 변수가 없으면 cell이 자동 skip된다.

**핵심 코드 3줄**:
- `pi_client.submit_document(pdf_path)` — 비동기 업로드, doc_id 반환
- `pi_client.get_document(doc_id)` — polling으로 상태 확인
- `pi_client.get_tree(doc_id, node_summary=True)` — 완성된 트리 JSON 받기

In [6]:
# ─────────────────────────────────────────────────────
# §6-1: sample_10k.pdf 업로드 + polling + 트리 페치
# ─────────────────────────────────────────────────────
doc_id_10k = None
pageindex_tree_10k = None

if HAVE_PAGEINDEX:
    print("sample_10k.pdf 업로드 중 ...")
    result = pi_client.submit_document(str(PDF_10K))
    doc_id_10k = result["doc_id"]
    print(f"  doc_id: {doc_id_10k}")

    # polling — 비동기 트리 빌드 완료 대기
    print("트리 빌드 polling ...")
    while True:
        status = pi_client.get_document(doc_id_10k).get("status")
        print(f"  status: {status}")
        if status == "completed":
            break
        elif status == "failed":
            print("  ✗ 실패"); break
        time.sleep(5)

    # 완성된 트리 페치 (node_summary=True로 summary 포함)
    tree_result = pi_client.get_tree(doc_id_10k, node_summary=True)
    pageindex_tree_10k = tree_result.get("result", [])
    print(f"\n✓ 트리 빌드 완료: 최상위 노드 {len(pageindex_tree_10k)}개")
else:
    print("PAGEINDEX_API_KEY 미설정 — cell skip")
    print()
    print("대안: Chapter 1의 tree_index.json을 PageIndex 트리 대용으로 사용")
    ch1_tree = WORK / "tree_index.json"
    if ch1_tree.exists():
        with open(ch1_tree) as f:
            ch1_data = json.load(f)
        print(f"  Chapter 1 트리 로드: {len(ch1_data.get('nodes', []))}개 노드")

sample_10k.pdf 업로드 중 ...
  doc_id: pi-cmpb1gdf400ai01pncgkrdqr6
트리 빌드 polling ...
  status: processing
  status: processing
  status: completed

✓ 트리 빌드 완료: 최상위 노드 3개


In [7]:
# ─────────────────────────────────────────────────────
# 트리 시각화 (들여쓰기로 계층 표시)
# ─────────────────────────────────────────────────────
def count_nodes(nodes):
    """재귀적으로 노드 수 카운트"""
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

def print_tree(nodes, indent=0):
    """들여쓰기로 트리 출력"""
    for n in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page = n.get("page_index", "?")
        print(f"{prefix}[{n['node_id']}] {n['title'][:50]}  (p.{page})")
        if n.get("nodes"):
            print_tree(n["nodes"], indent + 1)

if pageindex_tree_10k:
    print(f"총 노드 수: {count_nodes(pageindex_tree_10k)}\n")
    print_tree(pageindex_tree_10k)
else:
    print("(트리 미생성 — PAGEINDEX_API_KEY 필요)")

총 노드 수: 3

[0000] FY2025 Annual Report — Sample Corp  (p.1)
[0001] Chapter 1. Revenue  (p.1)
[0002] Chapter 2. Risk Factors  (p.1)


---
## §6-2 LLM Tree Search 분해

PageIndex의 retrieval은 임베딩 유사도가 아닌 LLM이 트리를 추론하여 node_list를 반환한다. 트리를 압축한 후 OpenAI에 JSON 응답을 강제한다.

이 셀은 트리만 있으면 동작한다 (PageIndex 키 없어도 Chapter 1 트리로 시연 가능). Chapter 4·5에서 직접 구현했던 패턴과 동일하다.

In [8]:
# ─────────────────────────────────────────────────────
# §6-2: LLM Tree Search 함수 정의
# ─────────────────────────────────────────────────────
def llm_tree_search(query: str, tree: list, model: str = "gpt-5.4-mini") -> dict:
    """트리 압축 + 프롬프트 → node_list 반환"""
    if not HAVE_OPENAI:
        return {"thinking": "OpenAI 키 없음", "node_list": []}

    # 트리 압축 — 150자 summary truncate + nested 유지
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150],     # 150자 절단
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out

    # 프롬프트 — JSON 응답 강제
    prompt = f"""You are given a query and a document's tree structure (like a TOC).
Identify which node IDs most likely contain the answer.

Query: {query}

Document Tree:
{json.dumps(compress(tree), indent=2)}

Reply ONLY as valid JSON:
{{
  "thinking": "<step-by-step reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},      # JSON 강제
        temperature=0,
    )
    return json.loads(response.choices[0].message.content)

### 6-2-1. 실행 — PageIndex 트리 또는 가상 트리로 시연

In [9]:
# ─────────────────────────────────────────────────────
# §6-2-1: 실행 — PageIndex 트리 또는 가상 트리
# ─────────────────────────────────────────────────────
# 트리가 없으면 §6-1 결과 또는 dummy 사용
if pageindex_tree_10k:
    test_tree = pageindex_tree_10k
elif HAVE_OPENAI:
    # 가상 트리로 시연 (PageIndex 키 없을 때)
    test_tree = [
        {"node_id":"r","title":"Sample 10-K","page_index":1,"text":"FY2025 Q1 results",
         "nodes":[
            {"node_id":"c1","title":"Revenue","page_index":1,
             "text":"iPhone 69702M, Mac 7744M, Services 26375M"},
            {"node_id":"c2","title":"Risk Factors","page_index":2,
             "text":"Supply chain disruption is the primary risk"},
        ]},
    ]
    print("PageIndex 트리 없음 → 가상 트리로 시연")
else:
    test_tree = []

# LLM Tree Search 실행
if test_tree and HAVE_OPENAI:
    result = llm_tree_search("FY2025 Q1 iPhone 매출은?", test_tree)
    print("쿼리: FY2025 Q1 iPhone 매출은?\n")
    print(f"Reasoning: {result['thinking'][:300]}")
    print(f"\n node_list: {result['node_list']}")

쿼리: FY2025 Q1 iPhone 매출은?

Reasoning: The query asks for FY2025 Q1 iPhone sales revenue. In the provided tree, the only section likely to contain revenue figures is Chapter 1. Revenue. The annual report root may also contain relevant high-level financial information, but the chapter on revenue is the most likely location.

 node_list: ['0001', '0000']


---
## §6-3 Expert-Guided Retrieval ★ (핵심)

PageIndex의 진짜 가치 — 임베딩 fine-tuning 없이 프롬프트 규칙만으로 도메인 적응. Chapter 4·5의 직접 구현에서는 다루지 않았던 패턴이며, 본 챕터에서 새롭게 학습하는 핵심이다.

### 6-3-1. Expert Rules 정의 — 영문 금융 + 한국어 공공

In [10]:
# ─────────────────────────────────────────────────────
# §6-3-1: 두 도메인의 Expert Rules
# ─────────────────────────────────────────────────────
FINANCIAL_RULES = """
Expert routing rules for financial documents (10-K):
- EBITDA·수익성 쿼리      → MD&A / Results of Operations
- 유동성·현금흐름 쿼리    → Cash Flow Statement
- 리스크 요인 쿼리        → Part I, Item 1A (Risk Factors)
- 매출 분해·세그먼트      → Segment reporting · Item 7
- 미래 전망·전략 쿼리      → CEO letter · Outlook section
- 부채·신용도 쿼리         → Balance Sheet · 부채 각주
- 규제·컴플라이언스 쿼리   → Legal Proceedings
"""

KOREAN_PUBLIC_RULES = """
한국 공공기관 보고서 라우팅 규칙:
- 매출·영업이익 쿼리       → 재무 정보 / 손익계산서 섹션
- 사업 영역 쿼리            → 회사 개요 / 사업 영역
- 환율·연료비 리스크 쿼리  → 리스크 요인 / 시장 리스크
- 탄소·ESG 쿼리            → 지속가능경영 / 규제 리스크
- 조직 구조 쿼리            → 회사 개요 / 조직 정보
- 부채·재무건전성 쿼리      → 재무 정보 / 부채 현황
"""

print("✓ Expert Rules 정의 완료 (영문 금융 + 한국어 공공)")

✓ Expert Rules 정의 완료 (영문 금융 + 한국어 공공)


### 6-3-2. Expert-Guided 검색 함수 — 라우팅 규칙을 프롬프트에 주입

without vs with 규칙의 node_list 차이를 정량 측정한다.

In [11]:
# ─────────────────────────────────────────────────────
# §6-3-2: Expert-Guided Retrieval 함수
# ─────────────────────────────────────────────────────
def llm_tree_search_with_expert(query, tree, expert_rules, model="gpt-5.4-mini"):
    """규칙 주입된 LLM Tree Search"""
    if not HAVE_OPENAI:
        return {"thinking": "OpenAI 키 없음", "node_list": []}

    def compress(nodes):
        out = []
        for n in nodes:
            entry = {"node_id":n["node_id"], "title":n["title"],
                     "page":n.get("page_index","?"),
                     "summary": n.get("text","")[:150]}
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out

    # ★ 핵심 — 프롬프트에 expert_rules 주입
    prompt = f"""You are a domain expert analyzing a document.
Find all node IDs that most likely contain the answer.
**Use the expert routing rules below to guide your reasoning.**

Query: {query}

Document Tree:
{json.dumps(compress(tree), indent=2)}

Expert Routing Rules (follow these carefully):
{expert_rules}

Reply ONLY as valid JSON:
{{
  "thinking": "<reasoning, referencing the expert rules>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = openai_client.chat.completions.create(
        model=model,
        messages=[{"role":"user", "content":prompt}],
        response_format={"type":"json_object"},
        temperature=0,
    )
    return json.loads(response.choices[0].message.content)

# Without vs With expert 비교
if test_tree and HAVE_OPENAI:
    query = "주요 리스크 요인은?"
    print(f"🔍 Query: {query}\n")

    print("── Without Expert Rules ──")
    basic = llm_tree_search(query, test_tree)
    print(f"  nodes: {basic['node_list']}")
    print(f"  count: {len(basic['node_list'])}")

    print("\n── With Expert Rules ──")
    guided = llm_tree_search_with_expert(query, test_tree, FINANCIAL_RULES)
    print(f"  nodes: {guided['node_list']}")
    print(f"  count: {len(guided['node_list'])}")
    print(f"  reasoning: {guided['thinking'][:200]}")

🔍 Query: 주요 리스크 요인은?

── Without Expert Rules ──
  nodes: ['0002']
  count: 1

── With Expert Rules ──
  nodes: ['0002']
  count: 1
  reasoning: The query asks for the main risk factors (주요 리스크 요인), which matches the expert routing rule for risk factor queries → Part I, Item 1A (Risk Factors). In the document tree, node 0002 is titled "Chapter


### 6-3-3. 한국어 PDF + 한국어 Expert Rules 적용

In [12]:
# ─────────────────────────────────────────────────────
# §6-3-3: 한국어 PDF → 트리 → 한국어 Expert Rules 적용
# ─────────────────────────────────────────────────────
doc_id_ko = None
pageindex_tree_ko = None

if HAVE_PAGEINDEX:
    print("kepco_ko_report.pdf 업로드 중 ...")
    result = pi_client.submit_document(str(PDF_KO))
    doc_id_ko = result["doc_id"]
    print(f"  doc_id: {doc_id_ko}")

    print("트리 빌드 polling ...")
    while True:
        status = pi_client.get_document(doc_id_ko).get("status")
        print(f"  status: {status}")
        if status == "completed": break
        elif status == "failed": print("✗ 실패"); break
        time.sleep(5)

    pageindex_tree_ko = pi_client.get_tree(doc_id_ko, node_summary=True).get("result", [])
    print(f"\n✓ 한국어 트리 노드 수: {count_nodes(pageindex_tree_ko)}")
    print_tree(pageindex_tree_ko[:5])

    # 한국어 Expert Rules로 retrieval
    print("\n── 한국어 공공문서 + Korean Rules ──")
    ko_result = llm_tree_search_with_expert(
        "환율 리스크가 사업에 미치는 영향은?",
        pageindex_tree_ko,
        KOREAN_PUBLIC_RULES,
    )
    print(f"  nodes: {ko_result['node_list']}")
    print(f"  reasoning: {ko_result['thinking'][:300]}")
else:
    print("Cloud 셀 skip (PAGEINDEX_API_KEY 미설정)")

kepco_ko_report.pdf 업로드 중 ...
  doc_id: pi-cmpb1ilxb00aj01pnmflhjl8g
트리 빌드 polling ...
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: completed

✓ 한국어 트리 노드 수: 7
[0000] تشرشل 2025ھ ھےھے  (p.1)
  └─ [0001] ھ 1 ھ ھے ھے  (p.1)
  └─ [0002] ھ 2 ھ ھے ھے  (p.1)
  └─ [0003] ہ 3 ہ ھے ھے  (p.1)
  └─ [0004] ہ 4 ہ ESG·ہنھے  (p.1)
    └─ [0005] 4.1 ہنہے ھے  (p.1)
    └─ [0006] 4.2 2050年  (p.2)

── 한국어 공공문서 + Korean Rules ──
  nodes: ['0004', '0005', '0006']
  reasoning: 쿼리는 '환율 리스크가 사업에 미치는 영향'으로, 전문가 라우팅 규칙상 '환율·연료비 리스크 쿼리 → 리스크 요인 / 시장 리스크'에 해당합니다. 따라서 사업에 영향을 주는 환율 관련 위험을 설명하는 리스크 요인 섹션이 가장 적합하며, 문서 트리에서는 해당 주제와 직접 연결될 가능성이 있는 상위/하위 노드를 우선 선택합니다.


---
## §6-4 Chat API + Multi-turn — Zero-Setup 챗봇

OpenAI API 키 없이 PageIndex 자체 LLM으로 multi-turn 대화. 챗봇 제품 빠른 프로토타이핑에 적합.

**한계 안내**: Chapter 5의 Verifier 같은 fail-closed 안전장치는 PageIndex Chat API에 내장되어 있지 않다. 정확도 critical 도메인은 별도 Verifier 추가가 필요하다.

In [13]:
# ─────────────────────────────────────────────────────
# §6-4: 단일 질문 — Chat API
# ─────────────────────────────────────────────────────
if HAVE_PAGEINDEX and doc_id_10k:
    print("=== 단일 질문 ===")
    response = pi_client.chat_completions(
        messages=[{"role":"user", "content":"FY2025 Q1 주요 매출 부문은?"}],
        doc_id=doc_id_10k,
    )
    answer = response["choices"][0]["message"]["content"]
    print(answer[:400])
else:
    print("Chat API cell skip (PAGEINDEX_API_KEY + doc_id 필요)")

=== 단일 질문 ===
FY2025 Q1의 주요 매출 부문은 다음과 같습니다:

| 부문 | Q1 매출 (백만 달러) | 전년 대비 성장률 |
|---|---|---|
| **iPhone** | $69,702M | +5.5% |
| **Mac** | $7,744M | +1.6% |
| **Services** | $26,375M | +11.5% |

- **iPhone**이 가장 큰 매출 비중을 차지하며, **Services(서비스)** 부문이 +11.5%로 가장 높은 성장률을 기록했습니다.
- Mac은 세 부문 중 가장 낮은 성장률(+1.6%)을 보였습니다.


In [14]:
# ─────────────────────────────────────────────────────
# Multi-turn 대화 — messages history 자동 관리
# ─────────────────────────────────────────────────────
if HAVE_PAGEINDEX and doc_id_10k:
    history = []
    questions = [
        "Q1 iPhone 매출은?",
        "Mac과 Services는?",
        "총합과 YoY 증감률은?",
    ]
    print("=== Multi-turn 대화 ===\n")
    for q in questions:
        history.append({"role":"user", "content":q})
        r = pi_client.chat_completions(messages=history, doc_id=doc_id_10k)
        reply = r["choices"][0]["message"]["content"]
        history.append({"role":"assistant", "content":reply})
        print(f"human: {q}")
        print(f"ai {reply[:250]}")
        print("-" * 55)
else:
    print("Cell skip")

=== Multi-turn 대화 ===

human: Q1 iPhone 매출은?
ai **Q1 iPhone 매출**은 **$69,702M (약 697억 달러)** 으로, 전년 동기 대비 **+5.5%** 성장했습니다.
-------------------------------------------------------
human: Mac과 Services는?
ai Q1 실적 기준:

- **Mac**: **$7,744M** (약 77억 달러), YoY **+1.6%**
- **Services**: **$26,375M** (약 264억 달러), YoY **+11.5%**

Services가 세 부문 중 가장 높은 성장률을 기록했습니다.
-------------------------------------------------------
human: 총합과 YoY 증감률은?
ai 앞선 답변들을 확인하기 위해 실제 문서 내용을 먼저 확인하겠습니다.문서에는 세그먼트별 수치만 있어 총합과 YoY는 계산값입니다:

| 구분 | Q1 Revenue (M$) |
|------|----------------|
| iPhone | 69,702 |
| Mac | 7,744 |
| Services | 26,375 |
| **합계** | **103,821** |

- **Q1 총 매출**: **$103,821M (약 1,038억 달러)**
-------------------------------------------------------


---
## §6-5 Managed vs Self-hosted 비교 (개념 시연)

실제 git clone 대신 두 모드의 차이를 표·코드로 정리한다 (full self-hosted 실행은 별도 환경에서 진행 권장).

In [15]:
# ─────────────────────────────────────────────────────
# §6-5: Self-hosted 실행 가이드
# ─────────────────────────────────────────────────────
print("=" * 60)
print("Self-hosted 실행 가이드")
print("=" * 60)
print()
print("# 1. 저장소 clone")
print("git clone https://github.com/VectifyAI/PageIndex.git")
print()
print("# 2. 의존성 설치")
print("cd PageIndex && pip install -r requirements.txt")
print()
print("# 3. .env 작성 (CHATGPT_API_KEY 주의 — OPENAI_API_KEY 아님)")
print("echo 'CHATGPT_API_KEY=$OPENAI_API_KEY' > .env")
print()
print("# 4. CLI 실행")
print("python run_pageindex.py \\")
print("    --pdf_path ./work/sample_10k.pdf \\")
print("    --model gpt-4o-2024-11-20 \\")
print("    --toc-check-pages 20 \\")
print("    --max-pages-per-node 10 \\")
print("    --if-add-node-summary yes")
print()
print("# 5. 결과: ./work/sample_10k_pageindex.json")

Self-hosted 실행 가이드

# 1. 저장소 clone
git clone https://github.com/VectifyAI/PageIndex.git

# 2. 의존성 설치
cd PageIndex && pip install -r requirements.txt

# 3. .env 작성 (CHATGPT_API_KEY 주의 — OPENAI_API_KEY 아님)
echo 'CHATGPT_API_KEY=$OPENAI_API_KEY' > .env

# 4. CLI 실행
python run_pageindex.py \
    --pdf_path ./work/sample_10k.pdf \
    --model gpt-4o-2024-11-20 \
    --toc-check-pages 20 \
    --max-pages-per-node 10 \
    --if-add-node-summary yes

# 5. 결과: ./work/sample_10k_pageindex.json
